# 02 - Persian Text Preprocessing
Turn raw Persian reviews into clean text a model can learn from:
normalization, noise removal, a **sentiment-aware** stopword list, and a fixed train / validation / test split.

## 0. Setup

In [ ]:
!pip install -q hazm

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
BASE_DIR = "/content/drive/MyDrive/snappfood-sentiment"
DATA_PATH = f"{BASE_DIR}/Snappfood - Sentiment Analysis.csv"
OUT_DIR = f"{BASE_DIR}/processed"
os.makedirs(OUT_DIR, exist_ok=True)

print("Data file found:", os.path.exists(DATA_PATH))

In [ ]:
import re
import warnings
from collections import Counter

import pandas as pd
from hazm import Normalizer, stopwords_list
from sklearn.model_selection import train_test_split

pd.set_option("display.max_colwidth", 150)
warnings.filterwarnings("ignore", message="This pattern")

## 1. Load data
Same as notebook 01. We already know all 70,000 rows are valid.

In [ ]:
raw = pd.read_csv(DATA_PATH, sep="\t", on_bad_lines="skip", engine="python")

df = raw[["comment", "label"]].copy()
df["label"] = df["label"].str.strip().str.upper()
df["sentiment"] = df["label"].map({"HAPPY": 1, "SAD": 0})
print(df.shape)
df.head()

## 2. What kind of noise do we have?
Before cleaning, let's count what we are actually dealing with.

In [ ]:
EMOJI_RE = re.compile("[\U0001F300-\U0001FAFF\U00002600-\U000027BF]")

checks = {
    "URL":                    r"https?://|www\.",
    "English letters":        r"[a-zA-Z]",
    "Digits (any script)":    r"[0-9\u06F0-\u06F9\u0660-\u0669]",
    "Arabic ي or ك":          r"[\u064A\u0643]",
    "Repeated letters (3+)":  r"(.)\1{2,}",
    "Emoji":                  EMOJI_RE.pattern,
}

noise = pd.DataFrame({
    name: [df["comment"].str.contains(p, regex=True).sum()] for name, p in checks.items()
}, index=["comments"]).T
noise["percent"] = (noise["comments"] / len(df) * 100).round(2)
noise

In [ ]:
# a few examples with repeated letters and emojis
print(df.loc[df["comment"].str.contains(r"(.)\1{2,}", regex=True), "comment"].head(5).to_string())
print()
print(df.loc[df["comment"].str.contains(EMOJI_RE.pattern, regex=True), "comment"].head(5).to_string())

## 3. The stopword trap
Hazm ships a ready-made Persian stopword list, but it was built for **news text**, not reviews.
Let's check whether it contains words that clearly carry sentiment.

In [ ]:
hazm_stopwords = set(stopwords_list())
print("hazm stopwords:", len(hazm_stopwords))

test_words = ["عالی", "خوب", "بهترین", "متاسفانه", "نبود", "نیست", "نه", "هیچ", "ولی", "خیلی", "کم"]
for w in test_words:
    print(f"{w:10} in hazm stopwords? {w in hazm_stopwords}")

In [ ]:
# how many reviews would lose an important word if we used the list as-is?
for w in ["عالی", "خوب", "نبود", "متاسفانه"]:
    n = df["comment"].str.contains(rf"\b{w}\b", regex=True).sum()
    print(f"{w:10} appears in {n:,} reviews")

Removing «عالی» (excellent) or «نبود» (wasn't) from a sentiment dataset would throw away the most useful signal.
So we **keep** negations, contrast words, intensifiers and evaluative words, and only remove the truly neutral ones.

In [ ]:
KEEP = {
    # negation
    "نه", "نبود", "نیست", "نیستند", "ندارد", "ندارند", "نباید", "نمی‌شود", "هیچ", "بدون", "عدم", "غیر", "بی",
    # contrast
    "اما", "ولی", "بلکه", "حتی", "فقط", "تنها",
    # intensity / amount
    "خیلی", "بسیار", "بسیاری", "کاملا", "کامل", "زیاد", "زیادی", "کم", "کمی", "بیش", "بیشتر", "پر", "کافی",
    # evaluation
    "عالی", "خوب", "خوبی", "بهتر", "بهترین", "مناسب", "متاسفانه", "جدی",
    # delivery / time (very relevant for food delivery)
    "رسید", "رسیدن", "دیر", "هنوز", "همیشه", "همواره", "دوباره",
}

STOPWORDS = hazm_stopwords - KEEP
print("final stopwords:", len(STOPWORDS))
print("rescued words:", sorted(hazm_stopwords & KEEP))

## 4. The cleaning function
Step by step:
1. lowercase English letters
2. remove URLs
3. hazm `Normalizer`: fixes Arabic ي/ك, spacing and half-spaces (نیم‌فاصله)
4. shorten repeated letters: «عالیییی» → «عالی»
5. keep emojis as separate tokens (😍 and 😡 are strong signals)
6. remove punctuation, digits and other symbols
7. drop stopwords and one-letter tokens

In [ ]:
normalizer = Normalizer()

URL_RE = re.compile(r"https?://\S+|www\.\S+")
REPEAT_RE = re.compile(r"(.)\1{2,}")
PERSIAN = "\u0621-\u063A\u0641-\u064A\u067E\u0686\u0698\u06A9\u06AF\u06CC\u200c"
EMOJI_CHARS = "\U0001F300-\U0001FAFF\U00002600-\U000027BF"
NOT_ALLOWED_RE = re.compile(f"[^{PERSIAN}{EMOJI_CHARS}a-z\\s]")

def clean_text(text, stopwords=STOPWORDS):
    text = str(text).lower()
    text = URL_RE.sub(" ", text)
    text = normalizer.normalize(text)
    text = REPEAT_RE.sub(r"\1", text)
    text = EMOJI_RE.sub(lambda m: f" {m.group(0)} ", text)
    text = NOT_ALLOWED_RE.sub(" ", text)

    tokens = []
    for tok in text.split():
        tok = tok.strip("\u200c")
        if not tok or tok in stopwords:
            continue
        if len(tok) == 1 and not EMOJI_RE.match(tok):
            continue
        tokens.append(tok)
    return " ".join(tokens)

In [ ]:
# test it on hand-written examples before running on all data
samples = [
    "غذا خيلي سرد بود و اصلا خوب نبود!!! 😡😡😡",
    "عالیییی بود ممنون از رستوران 😍",
    "پیتزا ۲ ساعت دیر رسید... www.snappfood.ir",
    "کیفیت غذا متوسط بود ولی پیک خیلی مودب بود",
]
for s in samples:
    print("before:", s)
    print("after: ", clean_text(s))
    print("naive: ", clean_text(s, stopwords=hazm_stopwords), "  <- with the original hazm list")
    print()

## 5. Clean all reviews

In [ ]:
%%time
df["clean_text"] = df["comment"].apply(clean_text)
df[["comment", "clean_text", "label"]].sample(10, random_state=1)

## 6. Check again: empty texts, duplicates, conflicts
After normalization, two reviews that looked different may now be identical.

In [ ]:
n0 = len(df)

empty = df["clean_text"].str.strip() == ""
print("empty after cleaning:", empty.sum())
df = df[~empty]

conflict = df.groupby("clean_text")["sentiment"].nunique()
conflict = conflict[conflict > 1].index
print("texts with conflicting labels:", len(conflict),
      "| rows:", df["clean_text"].isin(conflict).sum())
df = df[~df["clean_text"].isin(conflict)]

dups = df.duplicated(subset="clean_text").sum()
print("duplicates after cleaning:", dups)
df = df.drop_duplicates(subset="clean_text").reset_index(drop=True)

print(f"\nrows: {n0:,} -> {len(df):,}")
print(df["label"].value_counts())

In [ ]:
# what did the conflicting ones look like?
raw_conf = raw.assign(clean_text=raw["comment"].apply(clean_text))
raw_conf = raw_conf[raw_conf["clean_text"].isin(conflict)].sort_values("clean_text")
raw_conf[["comment", "label", "clean_text"]].head(10)

## 7. Frequent words and word pairs after cleaning

In [ ]:
def top_tokens(texts, n=20):
    c = Counter(t for txt in texts for t in txt.split())
    return pd.Series([w for w, _ in c.most_common(n)])

def top_bigrams(texts, n=20):
    c = Counter()
    for txt in texts:
        toks = txt.split()
        c.update(" ".join(p) for p in zip(toks, toks[1:]))
    return pd.Series([w for w, _ in c.most_common(n)])

happy = df.loc[df["label"] == "HAPPY", "clean_text"]
sad = df.loc[df["label"] == "SAD", "clean_text"]

pd.DataFrame({
    "HAPPY words": top_tokens(happy), "SAD words": top_tokens(sad),
    "HAPPY pairs": top_bigrams(happy), "SAD pairs": top_bigrams(sad),
})

In [ ]:
# the negation question from notebook 01: where does «خوب» appear in SAD reviews?
sad_good = df[(df["label"] == "SAD") & df["clean_text"].str.contains(r"\bخوب\b", regex=True)]
print("SAD reviews containing «خوب»:", len(sad_good))
print("...followed by «نبود»:", sad_good["clean_text"].str.contains("خوب نبود").sum())
sad_good[["comment", "clean_text"]].head(8)

## 8. Train / validation / test split
We split **once** and save the files, so every model in the next notebooks is trained and tested on exactly the same data.
- train 80% — to fit models
- validation 10% — to tune settings
- test 10% — touched only once, at the very end

`stratify` keeps the 50/50 label balance in each part.

In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df["sentiment"], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df["sentiment"], random_state=42)

cols = ["comment", "clean_text", "label", "sentiment"]
for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    part[cols].to_csv(f"{OUT_DIR}/{name}.csv", index=False)
    print(f"{name:5} {len(part):6,} rows | positive share: {part['sentiment'].mean():.3f}")

print("\nSaved to:", OUT_DIR)

## 9. Key findings
*(fill in after running)*
- 
- 
- 